# Chapitre 11 - Cours 2 : les courtiers fondés sur un journal partitionné

Un courtier de style AMQP/JMS hérite d'une mentalité de messagerie **transitoire** :
même s'il écrit sur disque, il efface le message dès qu'il est acquitté. Recevoir un
message est donc une opération **destructive** : on ne peut pas relancer un consommateur
pour obtenir le même résultat, et un nouveau consommateur ne voit que les messages
arrivés après son inscription. Les bases de données et les fichiers font l'inverse :
ce qui est écrit reste, et un nouveau client peut lire le passé. Les courtiers à
**journal** (log-based message brokers) combinent les deux : stockage durable d'une base,
notification à faible latence d'une messagerie.

Ce notebook couvre la section *Partitioned Logs* : *Using logs for message
storage*, *Logs compared to traditional messaging*, *Consumer offsets*, *Disk space
usage*, *When consumers cannot keep up with producers*, *Replaying old messages*. Les
exemples s'appuient sur la classe `PartitionedLog` de
[`ddia/streams.py`](../../ddia/streams.py), un journal partitionné en mémoire qui reprend
les choix de Kafka (y compris son partitionneur murmur2) ; le lab Kafka du chapitre
reproduit les mêmes expériences sur un vrai courtier.

**Plan.**

1. Un journal pour stocker les messages.
2. Journal contre messagerie traditionnelle : groupes de consommateurs et blocage en tête de file.
3. Les offsets des consommateurs.
4. Espace disque, tampon circulaire et consommateurs lents.
5. Rejouer les anciens messages.

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from ddia import streams as st
from ddia.diagrams import Diagram
from ddia.plotting import (AMBER, GRAY, GREEN, INDIGO, LIGHT, ROSE, SKY, SLATE, TEAL, VIOLET,
                           annotate_bars, finish, set_style, subtitle)

set_style()
plt.rcParams["axes.titlepad"] = 16
pd.set_option("display.precision", 2)
rng = np.random.default_rng(0)